In [ ]:
!pip install -q torchxrayvision albumentations

# ============================================================================
# REBUTTAL EXPERIMENT: Grad-CAM from denseblock3 (14x14) on the 21 epoch-25
# checkpoints. Requested by reviewers mtaQ, cHWi, fT7X. No retraining.
#
#   LAYER = "final"        -> 7x7, must reproduce the paper (mean 0.290,
#                             spread 0.0243, CI [0.0144, 0.0309]). Run FIRST.
#   LAYER = "denseblock3"  -> 14x14, the rebuttal number.
# ============================================================================

import os, json, time, zipfile
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import albumentations as A
import torchxrayvision as xrv
from sklearn.metrics import average_precision_score
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# ------------------------------- CONFIG -------------------------------------
LAYER     = "denseblock3"            # run "final" first, then "denseblock3"
ROOT      = "/content/drive/MyDrive/AI_Research/Model_Results/SaliencyRun/xrv_pipeline_nb"
REF_DIR   = f"{ROOT}/grad_vanilla_npy_25"            # used ONLY to get the 601 patient IDs + seed list
LABELS    = "/content/drive/MyDrive/AI_Research/stage_2_train_labels.csv"
ZIP_PATH  = "/content/drive/MyDrive/AI_Research/rsna-pneumonia-dataset-npy.zip"
NPY_MAP   = "/content/drive/MyDrive/AI_Research/npy_map.json"
GC_T1     = f"{ROOT}/paper_outputs/T1_per_seed.csv"  # paper's 7x7 per-seed means
OUT_NPY   = f"{ROOT}/gradcam_{LAYER}_npy_25"
OUT_DIR   = f"{ROOT}/gradcam_metrics_25_{LAYER}"
GC_CI     = (0.0144, 0.0309)
SCORE     = 1024
N_BOOT    = 1000
PNEUMONIA_LOGIT = 1
XRV_WEIGHTS = "densenet121-res224-mimic_nb"
DROPOUT     = 0.3366734543720239
IMG_SIZE    = 224
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
# ----------------------------------------------------------------------------

os.makedirs(OUT_NPY, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
t0 = time.time()

# ---- 0. seeds + patient IDs come from the existing M1 folders (guarantees same 601) ----
seeds = sorted(int(d.split("_")[1]) for d in os.listdir(REF_DIR) if d.startswith("seed_"))
assert len(seeds) == 21, f"found {len(seeds)} seeds, need 21 (pre-reg section 4)"
patient_ids = sorted(f[:-4] for f in os.listdir(f"{REF_DIR}/seed_{seeds[0]}") if f.endswith(".npy"))
assert len(patient_ids) == 601
print(f"21 seeds, 601 patients, layer={LAYER}, device={DEVICE}")

# ---- dataset: same zip + npy_map the export notebook used ----
if not os.path.exists("/content/rsna-pneumonia-dataset-npy"):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content")
with open(NPY_MAP) as f:
    npy_map = json.load(f)
missing = [p for p in patient_ids if p not in npy_map]
assert not missing, f"{len(missing)} patients missing from npy_map"

val_transform_xrv = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])


# ---- model + input, copied from the Grad-CAM export notebook ----
class XRVClassifier(nn.Module):
    def __init__(self, xrv_weights, dropout, num_classes=2):
        super().__init__()
        backbone = xrv.models.DenseNet(weights=xrv_weights)
        self.features = backbone.features
        num_features = backbone.classifier.in_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(num_features, num_classes))
    def forward(self, x):
        feats = torch.relu(self.features(x))
        return self.classifier(feats.mean(dim=(2, 3)))

def build_model_for_seed(seed):
    path = f"{ROOT}/frozen_xrv_seed{seed}_epoch25.pth"
    model = XRVClassifier(XRV_WEIGHTS, DROPOUT).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location="cpu"), strict=True)
    return model.eval()

def load_input(pid):
    img = np.load(npy_map[pid]).astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = (img * 255.0).clip(0, 255).astype(np.uint8)
    img = val_transform_xrv(image=img)["image"].astype(np.float32)
    img = xrv.datasets.normalize(img, 255)
    return torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()


class GradCAM:
    def __init__(self, model):
        self.model = model.eval(); self.acts = None; self.grads = None; self.h = None
        if LAYER == "denseblock3":
            self.h = model.features.denseblock3.register_forward_hook(self._fwd)
        elif LAYER != "final":
            raise ValueError(LAYER)
    def _fwd(self, _m, _i, out):
        self.acts = out; out.register_hook(lambda g: setattr(self, "grads", g))
    def __call__(self, x):
        self.model.zero_grad(set_to_none=True)
        if LAYER == "final":
            # post-ReLU 7x7 features, exactly what the paper's export used
            acts = torch.relu(self.model.features(x)); acts.retain_grad()
            logits = self.model.classifier(acts.mean(dim=(2, 3)))
            logits[0, PNEUMONIA_LOGIT].backward()
            grads = acts.grad
        else:
            logits = self.model(x)
            logits[0, PNEUMONIA_LOGIT].backward()
            acts, grads = self.acts, self.grads
        with torch.no_grad():
            w = grads.mean(dim=(2, 3), keepdim=True)
            cam = (w * acts).sum(dim=1)                 # raw; scorer rectifies + normalizes
        return cam[0].cpu().numpy().astype(np.float32)
    def close(self):
        if self.h is not None: self.h.remove()


# ---- 1. EXPORT: one .npy per (seed, patient), skip if already there ----
for i, s in enumerate(seeds, 1):
    d = f"{OUT_NPY}/seed_{s}"; os.makedirs(d, exist_ok=True)
    todo = [p for p in patient_ids if not os.path.exists(f"{d}/{p}.npy")]
    if not todo:
        print(f"[{i:2d}/21] seed {s:<5d} already exported"); continue
    model = build_model_for_seed(s); cam_fn = GradCAM(model)
    for pid in todo:
        np.save(f"{d}/{pid}.npy", cam_fn(load_input(pid).to(DEVICE)))
    cam_fn.close(); del model, cam_fn; torch.cuda.empty_cache()
    print(f"[{i:2d}/21] seed {s:<5d} exported {len(todo)} maps  ({time.time()-t0:.0f}s)")
shape = np.load(f"{OUT_NPY}/seed_{seeds[0]}/{patient_ids[0]}.npy").shape
print("raw map shape:", shape, "<- expect (14,14) for denseblock3, (7,7) for final")


# ---- 2. masks (identical to m1_scoring_cell) ----
lab = pd.read_csv(LABELS)
lab = lab[lab["Target"] == 1].dropna(subset=["x", "y", "width", "height"])
masks = {}
for pid in patient_ids:
    m = np.zeros((SCORE, SCORE), dtype=bool)
    for _, r in lab[lab["patientId"] == pid].iterrows():
        x0, y0 = int(round(r.x)), int(round(r.y))
        x1, y1 = int(round(r.x + r.width)), int(round(r.y + r.height))
        m[y0:y1, x0:x1] = True
    assert m.any(), pid
    masks[pid] = m


# ---- 3. scorer (identical to m1_scoring_cell / pre-reg section 5) ----
def score_map(raw, mask):
    a = np.maximum(raw.astype(np.float32), 0.0)
    a = a / max(a.max(), 1e-8)
    up = F.interpolate(torch.from_numpy(a)[None, None], size=(SCORE, SCORE),
                       mode="bilinear", align_corners=False)[0, 0].numpy()
    ap = average_precision_score(mask.ravel(), up.ravel())
    py, px = np.unravel_index(np.argmax(up), up.shape)
    return ap, bool(mask[py, px])


# ---- 4. score ----
per_seed, per_image = [], {p: [] for p in patient_ids}
for i, s in enumerate(seeds, 1):
    aps, hits = [], []
    for pid in patient_ids:
        ap, hit = score_map(np.load(f"{OUT_NPY}/seed_{s}/{pid}.npy"), masks[pid])
        aps.append(ap); hits.append(hit); per_image[pid].append(ap)
    per_seed.append({"seed": s, "auprc_mean": float(np.mean(aps)), "pointing_hit": float(np.mean(hits))})
    print(f"[{i:2d}/21] seed {s:<5d} AUPRC {np.mean(aps):.4f}  pointing {np.mean(hits):.3f}")


# ---- 5. aggregate ----
means = np.array([r["auprc_mean"] for r in per_seed])
spread = float(np.std(means, ddof=1))
rng = np.random.default_rng(0)
boots = [np.std(rng.choice(means, len(means), replace=True), ddof=1) for _ in range(N_BOOT)]
ci = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)))
img_sd = np.array([np.std(v, ddof=1) for v in per_image.values()])

gc7 = pd.read_csv(GC_T1).set_index("seed")["auprc_mean"]
paired = means - gc7.loc[seeds].values           # per-seed change vs the paper's 7x7

result = {
    "layer": LAYER, "raw_map_shape": list(shape), "n_seeds": 21, "n_images": 601,
    "mean_across_seeds": float(means.mean()),
    "per_seed_min": float(means.min()), "per_seed_max": float(means.max()),
    "spread_sd": spread, "spread_ci95": list(ci),
    "paper_7x7_reference": {"mean": 0.290, "spread": 0.0243, "ci95": list(GC_CI)},
    "per_image_sd_median": float(np.median(img_sd)),
    "per_image_sd_iqr": [float(np.percentile(img_sd, 25)), float(np.percentile(img_sd, 75))],
    "frac_images_sd_gt_0.02": float((img_sd > 0.02).mean()),
    "frac_images_sd_gt_0.11": float((img_sd > 0.11).mean()),
    "pointing_mean": float(np.mean([r["pointing_hit"] for r in per_seed])),
    "vs_paper_7x7_per_seed": {"mean_change": float(paired.mean()),
                              "seeds_improved": int((paired > 0).sum()),
                              "max_abs_diff": float(np.abs(paired).max()),
                              "rank_spearman": float(pd.Series(means).corr(pd.Series(gc7.loc[seeds].values), method="spearman"))},
}
pd.DataFrame(per_seed).to_csv(f"{OUT_DIR}/per_seed.csv", index=False)
json.dump(result, open(f"{OUT_DIR}/aggregate.json", "w"), indent=2)
print("\n" + json.dumps(result, indent=2))
print(f"\nwrote {OUT_DIR}/  ({time.time()-t0:.0f}s total)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 62.7 MB/s eta 0:00:00
Mounted at /content/drive
21 seeds, 601 patients, layer=denseblock3, device=cuda
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/mimic_nb-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/mimic_nb-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
[ 1/21] seed 7     exported 601 maps  (290s)
[ 2/21] seed 13    exported 601 maps  (330s)
[ 3/21] seed 21    exported 601 maps  (369s)
[ 4/21] seed 42    exported 601 maps  (408s)
[ 5/21] seed 99    exported 601 maps  (447s)
[ 6/21] seed 123   exported 601 maps  (487s)
[ 7/21] seed 256   exported 601 maps  (525s)
[ 8/21] seed 314   exported 601 maps  (564s)
[ 9/21] seed 500   exported 601 maps  (603s)
[10/21] seed 739   exported 601 maps  (642s)
[11/21] seed 777   exported 601 maps  (681s)
[12/21] seed 1000  exported 6